# Merge mathqaf datasets

Fusionne :
- `mathqaf_bulk_master.csv` (artist, origin, title, year, medium, post_code) — 1270 lignes parsées
- `mathqaf_media_full.csv` (code, caption, image_url, taken_at) — 3475 lignes brutes
- `mathqaf_images.zip` (images téléchargées, nommées `{code}.jpg`)

sur la clé commune `post_code` / `code`, et ajoute le chemin local de l'image quand elle existe.

Upload les 3 fichiers quand la cellule ci-dessous te le demande.

In [ ]:
from google.colab import files

print("Upload mathqaf_bulk_master.csv, mathqaf_media_full.csv, et mathqaf_images.zip (les 3 ensemble)")
uploaded = files.upload()
print(list(uploaded.keys()))

Upload mathqaf_bulk_master.csv, mathqaf_media_full.csv, et mathqaf_images.zip (les 3 ensemble)


KeyboardInterrupt: 

In [ ]:
import zipfile
import os

IMAGES_DIR = "images"

zip_name = next((n for n in uploaded if n.endswith(".zip")), None)
if zip_name:
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(IMAGES_DIR)
    print(f"Images extraites dans ./{IMAGES_DIR}")
else:
    print("Pas de zip trouvé dans l'upload, on suppose que le dossier images existe déjà.")

# gère le cas où le zip contient déjà un sous-dossier mathqaf_images/
for root, dirs, fnames in os.walk(IMAGES_DIR):
    jpgs = [f for f in fnames if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if jpgs:
        IMAGES_DIR = root
        break

print(f"Dossier images utilisé : {IMAGES_DIR}")
print(f"Nombre de fichiers image trouvés : {len([f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))])}")

In [ ]:
import pandas as pd

bulk_name = next(n for n in uploaded if "bulk_master" in n)
media_name = next(n for n in uploaded if "media_full" in n)

bulk = pd.read_csv(bulk_name)
media = pd.read_csv(media_name)

print("bulk_master:", bulk.shape, list(bulk.columns))
print("media_full:", media.shape, list(media.columns))

In [ ]:
# jointure sur post_code == code
merged = media.merge(
    bulk,
    left_on="code",
    right_on="post_code",
    how="left",  # garde toutes les 3475 lignes media_full, complète avec artist/title/year/medium quand dispo
)

# chemin image locale si le fichier existe
def local_path(code):
    p = os.path.join(IMAGES_DIR, f"{code}.jpg")
    return p if os.path.exists(p) else ""

merged["local_image_path"] = merged["code"].apply(local_path)

print("Total lignes :", len(merged))
print("Avec image locale téléchargée :", (merged["local_image_path"] != "").sum())
print("Avec artist/title parsés (from bulk_master) :", merged["artist"].notna().sum())

merged.to_csv("mathqaf_merged.csv", index=False)
merged.head(10)

In [ ]:
files.download("mathqaf_merged.csv")